<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day11-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 11 — Discussion 3: how much of the N-terminus does BiLSTM actually need? {.unnumbered}

The main notebook always looks at the first 70 residues (`NTERM_LEN=70`). Signal peptides are typically only 15-30 residues long. This notebook asks: does BiLSTM's advantage survive if the model only ever sees a much shorter N-terminal window — and does giving it *more* context past 70 residues help at all, given the rest of the protein carries no signal-peptide signal by definition?

In [1]:
import re
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, matthews_corrcoef, roc_auc_score

torch.manual_seed(0)
np.random.seed(0)

AA = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {a: i for i, a in enumerate(AA)}
UNK_IDX = len(AA)
VOCAB_SIZE = len(AA) + 1

def aa_index(ch):
    return aa_to_idx.get(ch, UNK_IDX)

def fetch_uniprot_tsv(query, fields, max_records=1200):
    base = "https://rest.uniprot.org/uniprotkb/search"
    params = {"query": query, "fields": fields, "format": "tsv", "size": 500}
    next_url = base + "?" + urllib.parse.urlencode(params)
    chunks, n_rows = [], 0
    while next_url and n_rows < max_records:
        req = urllib.request.Request(next_url, headers={"User-Agent": "EkmanTeaching/1.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            text = resp.read().decode("utf-8", errors="ignore")
            lines = text.strip().splitlines()
            if not lines:
                break
            if not chunks:
                chunks.extend(lines); n_rows += max(0, len(lines) - 1)
            else:
                chunks.extend(lines[1:]); n_rows += len(lines) - 1
            link = resp.headers.get("Link", "")
            m = re.search(r"<([^>]+)>;\s*rel=\"next\"", link)
            next_url = m.group(1) if m else None
    return "\n".join(chunks) + "\n"

def parse_signal_ranges(signal_field):
    if signal_field is None:
        return []
    txt = str(signal_field).strip()
    if txt == "" or txt.lower() == "nan":
        return []
    ranges = []
    for m in re.finditer(r"SIGNAL\s+(\d+)\.\.(\d+)", txt):
        ranges.append((int(m.group(1)), int(m.group(2))))
    return ranges

def load_dataset(query, fields="accession,sequence,ft_signal,organism_name", max_records=1200):
    tsv_text = fetch_uniprot_tsv(query, fields, max_records)
    rows = [r.split("\t") for r in tsv_text.strip().splitlines()]
    header, data = rows[0], rows[1:]
    sequences, labels = [], []
    for row in data:
        rec = dict(zip(header, row))
        seq = rec.get("Sequence", "")
        if not seq:
            continue
        ranges = parse_signal_ranges(rec.get("Signal peptide"))
        yseq = [0] * len(seq)
        for (a, b) in ranges:
            for p in range(a - 1, min(b, len(seq))):
                yseq[p] = 1
        sequences.append(seq)
        labels.append(yseq)
    return sequences, labels

def make_seq_tensors(seq_list, lab_list, nterm_len=70):
    X = np.full((len(seq_list), nterm_len), UNK_IDX, dtype=np.int64)
    Y = np.zeros((len(seq_list), nterm_len), dtype=np.float32)
    M = np.zeros((len(seq_list), nterm_len), dtype=np.float32)
    for i, (seq, yseq) in enumerate(zip(seq_list, lab_list)):
        L = min(len(seq), nterm_len)
        for j in range(L):
            X[i, j] = aa_index(seq[j])
            Y[i, j] = float(yseq[j])
            M[i, j] = 1.0
    return X, Y, M

class SeqTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim=24, hid_dim=48, kind="lstm", num_layers=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        bidirectional = (kind == "bilstm")
        if kind in ("lstm", "bilstm"):
            self.rnn = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
        elif kind == "gru":
            self.rnn = nn.GRU(emb_dim, hid_dim, num_layers=num_layers, batch_first=True)
        else:
            self.rnn = nn.RNN(emb_dim, hid_dim, num_layers=num_layers, batch_first=True, nonlinearity="tanh")
        out_dim = hid_dim * (2 if bidirectional else 1)
        self.out = nn.Linear(out_dim, 1)

    def forward(self, x):
        z = self.emb(x)
        h, _ = self.rnn(z)
        return self.out(h).squeeze(-1)

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def train_model(model, train_loader, epochs=12, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.BCEWithLogitsLoss(reduction="none")
    for ep in range(epochs):
        model.train()
        for Xb, Yb, Mb in train_loader:
            opt.zero_grad()
            logits = model(Xb)
            loss = (lossf(logits, Yb) * Mb).sum() / Mb.sum()
            loss.backward()
            opt.step()
    return model

@torch.no_grad()
def evaluate(model, X_test, Y_test, M_test):
    model.eval()
    logits = model(torch.from_numpy(X_test))
    proba = torch.sigmoid(logits).numpy()
    mask = M_test.astype(bool)
    y_true = Y_test[mask]
    p = proba[mask]
    pred = (p >= 0.5).astype(np.int64)
    return {
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "auroc": roc_auc_score(y_true, p),
    }


## 1) Refetch once, re-window at three different lengths

In [2]:

sequences, labels = load_dataset("reviewed:true AND length:[60 TO 800]", max_records=900)
idx = np.arange(len(sequences))
rng = np.random.default_rng(0)
rng.shuffle(idx)
split = int(0.8 * len(idx))
seq_train = [sequences[i] for i in idx[:split]]; lab_train = [labels[i] for i in idx[:split]]
seq_test = [sequences[i] for i in idx[split:]]; lab_test = [labels[i] for i in idx[split:]]
print(f"train={len(seq_train)} test={len(seq_test)}")


train=800 test=200


## 2) BiLSTM at N-terminal window lengths 20, 70, and 150

In [3]:

for nterm in [20, 70, 150]:
    X_train, Y_train, M_train = make_seq_tensors(seq_train, lab_train, nterm_len=nterm)
    X_test, Y_test, M_test = make_seq_tensors(seq_test, lab_test, nterm_len=nterm)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train), torch.from_numpy(M_train)), batch_size=64, shuffle=True)
    torch.manual_seed(0)
    model = SeqTagger(VOCAB_SIZE, kind="bilstm")
    train_model(model, train_loader, epochs=12)
    metrics = evaluate(model, X_test, Y_test, M_test)
    print(f"nterm_len={nterm:>4d} -> F1={metrics['f1']:.3f} MCC={metrics['mcc']:.3f} AUROC={metrics['auroc']:.3f}")


nterm_len=  20 -> F1=0.807 MCC=0.762 AUROC=0.948


nterm_len=  70 -> F1=0.838 MCC=0.825 AUROC=0.981


nterm_len= 150 -> F1=0.753 MCC=0.745 AUROC=0.989


## Try it yourself

Run the same three window lengths with `kind="lstm"` (forward-only) instead of `"bilstm"` — does the forward-only model lose *more* accuracy than BiLSTM does when the window shrinks to 20, since it has even less context to work with in each direction?